# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
aviation_df = pd.read_csv("data/AviationData.csv", encoding="latin-1")

print(f"Inspect NaNs:\n{aviation_df.isna().sum()}")
print(f"Inspect datatypes:\n{aviation_df.dtypes}")
print(f"Summary statistics:\n{aviation_df.describe()}")
print(f"Shape:\n{aviation_df.shape}")

Inspect NaNs:
Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
Publication.Date          

/var/folders/y_/8lgqgn_17j53v04m52jxzw7c0000gn/T/ipykernel_75106/2155146299.py:1: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  aviation_df = pd.read_csv("data/AviationData.csv", encoding="latin-1")


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [3]:
print(f"Aircraft.Category unique values:\n{aviation_df["Aircraft.Category"].unique()}")
print(f"Amateur.Built unique values:\n{aviation_df["Amateur.Built"].unique()}")

print(f"Aircraft.Category mode:\n{aviation_df["Aircraft.Category"].mode()}")
# The most common aircraft category is airplane, to find out if this is a reasonable imputation, we'll look at some nan/Unknown rows.
print(f"Unknown/NaN categorized aircrafts:\n{aviation_df.loc[(aviation_df["Aircraft.Category"] == "Unknown") | (aviation_df["Aircraft.Category"].isna()), ["Aircraft.Category", "Make", "Model"]].sample(5)}")
# From checking the makes and models, most are airplanes. This is a reasonable imputation.

aviation_df["Aircraft.Category"] = aviation_df["Aircraft.Category"].replace(["Unknown", "UNK"], "Airplane")
aviation_df["Aircraft.Category"] = aviation_df["Aircraft.Category"].fillna("Airplane")

# Filter for airplanes, professionally-built, and data from 1983 onwards
aviation_df = aviation_df[aviation_df["Aircraft.Category"] == "Airplane"]
aviation_df = aviation_df[aviation_df["Amateur.Built"] == "No"]
aviation_df = aviation_df[pd.to_datetime(aviation_df["Event.Date"]).dt.year >= 1983]

print(f"Shape:\n{aviation_df.shape}")

Aircraft.Category unique values:
[nan 'Airplane' 'Helicopter' 'Glider' 'Balloon' 'Gyrocraft' 'Ultralight'
 'Unknown' 'Blimp' 'Powered-Lift' 'Weight-Shift' 'Powered Parachute'
 'Rocket' 'WSFT' 'UNK' 'ULTR']
Amateur.Built unique values:
['No' 'Yes' nan]
Aircraft.Category mode:
0    Airplane
Name: Aircraft.Category, dtype: object
Unknown/NaN categorized aircrafts:
      Aircraft.Category         Make   Model
10062               NaN     Bellanca     7EC
44814               NaN       Cessna   A185F
30146               NaN       Cessna   A188B
23607               NaN       Cessna     172
28671               NaN  Taylorcraft  BC-12D
Shape:
(73015, 31)


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [4]:
# Injury.Severity records either fatal or non-fatal, but does not count serious injuries, so the Total.Fatal.Injuries and Total.Serious.Injuries must be added manually.

# Clean passenger condition data, then sum passenger conditions across the columns and store to "Passengers" column.
print(f'Passenger condition mode:\n{aviation_df.loc[:, "Total.Fatal.Injuries":"Total.Uninjured"].mode()}')
# The mode of each passenger condition is 0, which is a reasonable imputation.
aviation_df.loc[:, "Total.Fatal.Injuries":"Total.Uninjured"] = aviation_df.loc[:, "Total.Fatal.Injuries":"Total.Uninjured"].fillna(0)
# Calculate number of passengers by summing passenger conditions.
aviation_df["Number.of.Passengers"] = aviation_df.loc[:, "Total.Fatal.Injuries":"Total.Uninjured"].sum(axis=1)

# Severity.Likelihood is a value [0:1]: sum of fatal and serious injuries divided by number of passengers.
aviation_df["Severity.Likelihood"] = aviation_df.loc[:, "Total.Fatal.Injuries":"Total.Serious.Injuries"].sum(axis=1) / aviation_df["Number.of.Passengers"]

# We are focused on commercial and passenger aircrafts, so drop where Number.of.Passengers = 0
aviation_df = aviation_df[aviation_df["Number.of.Passengers"] > 0]
# Since we have Number.of.Passengers > 0, Severity.Likelihood != NaN
print(f"Severity.Likelihood NaN:{aviation_df["Severity.Likelihood"].isna().sum()}")

print(f"Shape:\n{aviation_df.shape}")

Passenger condition mode:
   Total.Fatal.Injuries  Total.Serious.Injuries  Total.Minor.Injuries  \
0                   0.0                     0.0                   0.0   

   Total.Uninjured  
0              0.0  
Severity.Likelihood NaN:0
Shape:
(71832, 33)


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [5]:
print(f"Aircraft damage mode:{aviation_df["Aircraft.damage"].mode()}")
# The most common amount of damage is Substantial. From earlier NaN inspection, there are only 3000 NaN (~4% of dataset), a very insignificant amount, so a better option may be to drop.
aviation_df = aviation_df.dropna(subset="Aircraft.damage")

# Construct "Aircraft.Destroyed"
aviation_df["Aircraft.Destroyed"] = aviation_df["Aircraft.damage"] == "Destroyed"

print(f"Shape:\n{aviation_df.shape}")

Aircraft damage mode:0    Substantial
Name: Aircraft.damage, dtype: object
Shape:
(69484, 34)


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [6]:
# The goal of cleaning here is to ensure that same Makes are considered the same regardless of user-input error or syntax inconsistencies.

# Drop NaN
aviation_df = aviation_df.dropna(subset="Make")

## Make each Make in UPPERCASE
aviation_df["Make"] = aviation_df["Make"].str.upper()

# Remove periods and commas
aviation_df["Make"] = aviation_df["Make"].str.replace('.', '')
aviation_df["Make"] = aviation_df["Make"].str.replace(',', '')

# Remove the suffix of each Make, as these are inconsistent in the dataset,
# and can cause same Make to be considered different
# Use regex $ so INC, LTD, etc. is only removed if at the end
aviation_df["Make"] = aviation_df["Make"].str.replace(r'\s*(INC|LTD|LLC|CORP|CORPORATION|AVIATION|COMPANY|CO|INCORPORATE|INCORPORATED)\.?$', '', regex=True)

# Strip whitespace from beginning and end of Make
aviation_df["Make"] = aviation_df["Make"].str.strip()

# Keep the Makes that have atleast 50 entries
make_counts = aviation_df["Make"].value_counts()
aviation_df = aviation_df[aviation_df["Make"].map(make_counts) >= 50]

print(f"Shape:\n{aviation_df.shape}")

Shape:
(64638, 34)


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [7]:
# Drop NaN
aviation_df = aviation_df.dropna(subset='Model')

# Models are NOT unique to each Make, shown by this example using Make "CESSNA"
print(f"Sample of different CESSNA Models:\n{aviation_df.loc[aviation_df["Make"] == "CESSNA", ["Make","Model"]].sample(5)}")

# Construct Aircraft.Type from Make and Model
aviation_df["Aircraft.Type"] = aviation_df["Make"] + ' ' + aviation_df["Model"]

print(f"Shape:\n{aviation_df.shape}")

Sample of different CESSNA Models:
         Make  Model
21649  CESSNA   172N
9492   CESSNA    172
24327  CESSNA   182D
76746  CESSNA   180B
5552   CESSNA  U206G
Shape:
(64624, 35)


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [8]:
print(f"Engine type unique values:{aviation_df["Engine.Type"].unique()}")
# Engine type is clean.

print(f"Weather condition unique values:{aviation_df["Weather.Condition"].unique()}")
# Weather condition has two types of unknowns (UNK and Unk), could be cleaned for consistency.
aviation_df["Weather.Condition"] = aviation_df["Weather.Condition"].str.upper()

print(f"Number of engines unique values:{aviation_df["Number.of.Engines"].unique()}")
# Number of engines is clean, ~6% NaN so could either be imputed or dropped without much effect.

print(f"Purpose of flight unique values:{aviation_df["Purpose.of.flight"].unique()}")
# Purpose of flight has many categories with some overlaps, could be cleaned for consistency.
aviation_df["Purpose.of.flight"] = aviation_df["Purpose.of.flight"].replace("Air Race show", "Air Race/show")

print(f"Broad phase of flight unique values:{aviation_df["Broad.phase.of.flight"].unique()}")
# Broad plan of flight has ~30% NaN. Since it is categorial with many categories, and with a significant amount of NaN, it should be dropped.
aviation_df = aviation_df.dropna(subset="Broad.phase.of.flight")

print(f"Shape:\n{aviation_df.shape}")

Engine type unique values:['Reciprocating' 'Turbo Prop' 'Turbo Fan' 'Turbo Shaft' 'Unknown'
 'Turbo Jet' nan 'Geared Turbofan' 'UNK']
Weather condition unique values:['VMC' 'IMC' 'UNK' nan 'Unk']
Number of engines unique values:[ 1.  2.  0.  4.  3. nan]
Purpose of flight unique values:['Personal' 'Instructional' 'Business' 'Executive/corporate' 'Unknown'
 'Aerial Observation' 'Public Aircraft' 'Aerial Application' 'Ferry' nan
 'Other Work Use' 'Positioning' 'Skydiving' 'Flight Test' 'Air Race/show'
 'Air Drop' 'Public Aircraft - Federal' 'Glider Tow'
 'Public Aircraft - Local' 'External Load' 'Public Aircraft - State'
 'Banner Tow' 'Firefighting' 'Air Race show' 'PUBS' 'ASHO']
Broad phase of flight unique values:['Approach' 'Landing' 'Takeoff' 'Maneuvering' 'Cruise' 'Climb' 'Go-around'
 'Taxi' 'Descent' nan 'Standing' 'Unknown' 'Other']
Shape:
(48388, 35)


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [9]:
print(f"Reinspect NaNs:\n{aviation_df.isna().sum()}")

# Drop NaNs with more than 50% unknowns.
aviation_df = aviation_df.dropna(thresh=len(aviation_df) * 0.5, axis=1)

print(f"Shape:\n{aviation_df.shape}")

Reinspect NaNs:
Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     12
Country                     174
Latitude                  39779
Longitude                 39784
Airport.Code              20657
Airport.Name              18958
Injury.Severity               0
Aircraft.damage               0
Aircraft.Category             0
Registration.Number           2
Make                          0
Model                         0
Amateur.Built                 0
Number.of.Engines           483
Engine.Type                 116
FAR.Description           45796
Schedule                  41783
Purpose.of.flight           771
Air.carrier               46183
Total.Fatal.Injuries          0
Total.Serious.Injuries        0
Total.Minor.Injuries          0
Total.Uninjured               0
Weather.Condition            38
Broad.phase.of.flight         0
Report.Status                 0
Publication.Date        

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [10]:
aviation_df.to_csv('Aviation_Cleaned.csv', index=False)